# 03 - Model Training & Experiment Tracking

This notebook covers:
1. Loading training data from the Feature Store dataset
2. Training an XGBoost classifier with cross-validation
3. Experiment tracking (logging params, metrics, model)
4. Model evaluation and feature importance analysis

**Training approach:**
- XGBoost with `scale_pos_weight` to handle class imbalance (3% fraud)
- 5-fold stratified cross-validation for robust evaluation
- Key metrics: AUC-ROC, PR-AUC, Precision, Recall, F1

In [ ]:
import sys
sys.path.insert(0, "../source")
from snowpark_session import create_snowpark_session
from config import DATABASE, SCHEMA, WAREHOUSE

session = create_snowpark_session()
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()
session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()

## Step 1: Load Training Data from Feature Store

In [ ]:
from training.train_model import load_training_data, FEATURE_COLUMNS, LABEL_COLUMN

df = load_training_data(session, DATABASE, SCHEMA)
print(f"\nDataset shape: {df.shape}")
print(f"Features available: {[c for c in FEATURE_COLUMNS if c in df.columns]}")
print(f"Label distribution:\n{df[LABEL_COLUMN].value_counts()}")

## Step 2: Train Model with Experiment Tracking

We use Snowflake's `ExperimentTracking` to log:
- Hyperparameters (n_estimators, learning_rate, etc.)
- Metrics per CV fold and on holdout test set
- The trained model artifact itself

In [ ]:
from training.train_model import train_model, save_model_locally
from snowflake.ml.tracking import ExperimentTracking

# Setup experiment tracking
experiment = ExperimentTracking(
    session=session,
    experiment_name="FRAUD_DETECTION_EXPERIMENTS",
    run_name="xgboost_v1_notebook",
    create_if_not_exists=True,
)
experiment.set_live_logging_status(True)

# Train with default hyperparameters
model, metrics, importance = train_model(
    df, params=None, experiment=experiment, run_name="xgboost_v1_notebook"
)

## Step 3: Evaluation & Quality Gate Check

In [ ]:
from training.evaluate_model import check_quality_gate
from config import PIPELINE_CONFIG

# Check against pipeline thresholds
passed, reason = check_quality_gate(metrics, PIPELINE_CONFIG)
print(f"Quality Gate: {'PASSED' if passed else 'FAILED'}")
print(f"Reason: {reason}")
print(f"\nMetrics vs Thresholds:")
print(f"  AUC-ROC:   {metrics['auc_roc']:.4f} (min: {PIPELINE_CONFIG['min_auc_roc']})")
print(f"  Precision: {metrics['precision']:.4f} (min: {PIPELINE_CONFIG['min_precision']})")
print(f"  Recall:    {metrics['recall']:.4f} (min: {PIPELINE_CONFIG['min_recall']})")

# Feature importance
print(f"\nTop 10 Features:")
for feat, imp in list(importance.items())[:10]:
    print(f"  {feat:30s} {imp:.4f}")

## Step 4: Save Model Locally

The model is serialized locally. In the next notebook, we'll register it to the Snowflake Model Registry for deployment.

In [ ]:
model_path = save_model_locally(model)
print(f"\nModel ready for registry registration.")
print(f"Next: Run notebook 04 to register and deploy.")